In [1]:
import numpy as np
import pandas as pd

# Functions

In [2]:
def quad_function(x, a=False):
    return -(1 if a else 2) * x**2 + 10

def cubic_function(x, a=False):
    return 3*x + x**3 * (0.2 if a else 0.02)

def sin_function(x, a=False):
    return np.sin(x + (-2 if a else 0) ) * 10

# Generate noise data

In [3]:
def generate_noise_data(function, n_samples, x_true_range, noise_std=1.2, seed=1, a_prob=0.5):
    np.random.seed(seed)

    X_true = np.random.uniform(*x_true_range, n_samples)
    x_noise = np.random.normal(0, noise_std, n_samples)

    X_obs = X_true + x_noise
    
    # Generate categorical feature 'a' (0 or 1)
    a_categorical = np.random.binomial(1, a_prob, n_samples)
    
    # Apply function with categorical feature
    y = np.array([function(x, bool(a)) for x, a in zip(X_true, a_categorical)])

    return pd.DataFrame({
        "X_obs": X_obs, 
        "X_true": X_true, 
        "x_noise": x_noise,
        "a": a_categorical,
        "y": y
    })

In [4]:
def generate_multi_environment_data(function, n_samples, train_range, test_range_1, test_range_2, 
                                   noise_std=1.2, n_environments=10, seed_base=1, a_prob=0.5):
    """
    Generiere Daten mit mehreren Environments durch Aufteilung des Train-Ranges
    """
    # Aufteilen des train_range in n_environments gleichmäßige Bereiche
    train_start, train_end = train_range
    range_width = (train_end - train_start) / n_environments
    env_ranges = [(train_start + i * range_width, train_start + (i + 1) * range_width) for i in range(n_environments)]
    
    all_data = []
    n_samples_per_env = n_samples // n_environments
    
    # Generiere Daten für jedes Environment
    for domain_id, env_range in enumerate(env_ranges):
        env_data = generate_noise_data(function, n_samples_per_env, env_range, 
                                     noise_std, seed=seed_base + domain_id, a_prob=a_prob)
        env_data["split"] = "train"
        env_data["domain_id"] = domain_id
        env_data["env_range_start"] = env_range[0]
        env_data["env_range_end"] = env_range[1]
        all_data.append(env_data)
    
    # Test-Daten
    test_data_1 = generate_noise_data(function, test_range_1[2] if len(test_range_1) > 2 else 500, 
                                     test_range_1[:2], noise_std, seed=seed_base + n_environments, a_prob=a_prob)
    test_data_1["split"] = "test"
    test_data_1["domain_id"] = -1  # -1 für Test-Daten
    test_data_1["env_range_start"] = test_range_1[0]
    test_data_1["env_range_end"] = test_range_1[1]
    all_data.append(test_data_1)
    
    test_data_2 = generate_noise_data(function, test_range_2[2] if len(test_range_2) > 2 else 500, 
                                     test_range_2[:2], noise_std, seed=seed_base + n_environments + 1, a_prob=a_prob)
    test_data_2["split"] = "test"
    test_data_2["domain_id"] = -1  # -1 für Test-Daten
    test_data_2["env_range_start"] = test_range_2[0]
    test_data_2["env_range_end"] = test_range_2[1]
    all_data.append(test_data_2)
    
    return pd.concat(all_data, ignore_index=True)

In [5]:
train_range = (-4, 4)
test_range_1 = (-6.5, -4)
test_range_2 = (4, 6.5)

n_train = 1500
n_test = 500

noise_std = 1.2
y_noise_std = 0.4

## Quad data

In [6]:
data = generate_multi_environment_data(
    function=quad_function,
    n_samples=n_train,
    train_range=train_range,
    test_range_1=test_range_1,
    test_range_2=test_range_2,
    noise_std=noise_std,
    n_environments=10,
    seed_base=1
)

data.to_csv("../data/quad_data.csv", index=False)

data.head()

,X_obs,X_true,x_noise,a,y,split,domain_id,env_range_start,env_range_end
0,-5.314123,-3.666382,-1.647741,0,-16.884720,train,0,-4.0,-3.2
1,-3.045549,-3.423740,0.378191,1,-1.721998,train,0,-4.0,-3.2
2,-2.984516,-3.999909,1.015393,1,-5.999268,train,0,-4.0,-3.2
3,-4.789553,-3.758134,-1.031419,0,-18.247141,train,0,-4.0,-3.2
4,-3.461940,-3.882595,0.420655,0,-20.149092,train,0,-4.0,-3.2


## Cubic data

In [7]:
data = generate_multi_environment_data(
    function=cubic_function,
    n_samples=n_train,
    train_range=train_range,
    test_range_1=test_range_1,
    test_range_2=test_range_2,
    noise_std=noise_std,
    n_environments=10,
    seed_base=1
)

data.to_csv("../data/cubic_data.csv", index=False)

data.head()

,X_obs,X_true,x_noise,a,y,split,domain_id,env_range_start,env_range_end
0,-5.314123,-3.666382,-1.647741,0,-11.984844,train,0,-4.0,-3.2
1,-3.045549,-3.423740,0.378191,1,-18.297837,train,0,-4.0,-3.2
2,-2.984516,-3.999909,1.015393,1,-24.798847,train,0,-4.0,-3.2
3,-4.789553,-3.758134,-1.031419,0,-12.335967,train,0,-4.0,-3.2
4,-3.461940,-3.882595,0.420655,0,-12.818353,train,0,-4.0,-3.2


## Sin function

In [8]:
data = generate_multi_environment_data(
    function=sin_function,
    n_samples=n_train,
    train_range=train_range,
    test_range_1=test_range_1,
    test_range_2=test_range_2,
    noise_std=noise_std * 0.5,
    n_environments=10,
    seed_base=1
)

data.to_csv("../data/sin_data.csv", index=False)

data.head()

,X_obs,X_true,x_noise,a,y,split,domain_id,env_range_start,env_range_end
0,-4.490253,-3.666382,-0.823870,0,5.010311,train,0,-4.0,-3.2
1,-3.234645,-3.423740,0.189096,1,7.574803,train,0,-4.0,-3.2
2,-3.492212,-3.999909,0.507696,1,2.795034,train,0,-4.0,-3.2
3,-4.273844,-3.758134,-0.515710,0,5.782167,train,0,-4.0,-3.2
4,-3.672268,-3.882595,0.210328,0,6.750280,train,0,-4.0,-3.2
